# **Multiplicative (Luong General) Attention**

This notebook picks up right where [8_Encoder_Decoder.ipynb](./8_Encoder_Decoder.ipynb) left off. That notebook ended with the Discussion pointing out the single-context-vector bottleneck, and how attention fixes it by letting the decoder look back at every encoder hidden state instead of just the last one.

We already saw two ways to *score* how relevant an encoder position is:

- **Bahdanau (additive)** - `score = Va(tanh(Wa·query + Ua·keys))`, a small feedforward network learns the scoring.
- **Luong dot** - `score = query · keys`, a raw dot product, no learned weights at all.

This notebook covers the third option: **Luong general (multiplicative) attention**, which sits in between the two, it keeps a learned weight matrix like Bahdanau, but scores with a matrix multiply instead of a feedforward network, which is why its called *multiplicative*.


## **What makes it "multiplicative"?**

The core scoring formula for multiplicative/general attention is:

```
score(s_t, h_i) = s_t^T · Wa · h_i
```

Where `s_t` is the decoder's current hidden state (the query) and `h_i` is an encoder hidden state (a key). Compare this to dot attention, `score = s_t^T · h_i`, theres no `Wa` at all there, its just a straight dot product.

Multiplicative attention slides a learned weight matrix `Wa` in between the two vectors before taking the dot product. What this buys us is that `query` and `keys` no longer have to line up in the same vector space directly, `Wa` learns a transformation that maps one into a space where dot-product similarity actually means something useful. Dot attention only works well when the query and key spaces already happen to align well on their own, general attention doesn't need that assumption, it learns the alignment.

Its still cheaper than Bahdanau though, one matrix multiply against `Wa` versus a full two-layer feedforward network (`Wa`, `Ua`, `Va`) squashed through a `tanh`. Roughly:

| | Bahdanau (Additive) | Luong Dot | Luong General (Multiplicative) |
|---|---|---|---|
| Formula | `Va(tanh(Wa·q + Ua·k))` | `q · k` | `q · Wa · k` |
| Learned parameters | `Wa`, `Ua`, `Va` | none | `Wa` |
| Cost | Highest (feedforward + tanh) | Lowest (just a dot product) | In between (one matrix multiply) |


## **Where attention sits relative to the RNN (this part actually matters)**

This is the part thats easy to get wrong if we just copy the Bahdanau decoder's structure and swap the attention class in, so lets be careful here.

**Bahdanau's decoder (the one we already built)** attends *before* the RNN step:

```
1. query = previous decoder hidden state (before this timestep's RNN call)
2. context = attention(query, encoder_outputs)
3. rnn_input = concat(embedded_word, context)
4. output, hidden = rnn(rnn_input, previous_hidden)
```

**Luong's decoder** attends *after* the RNN step instead:

```
1. rnn_output, hidden = rnn(embedded_word, previous_hidden)   # RNN runs first, plain embedding, no context yet
2. query = rnn_output   # this timestep's fresh RNN output becomes the query
3. attentional_hidden, weights = attention(query, encoder_outputs)
4. output = out(attentional_hidden)   # prediction comes from this fused vector, not straight from rnn_output
```

Notice the RNN in Luong's version only ever takes the plain embedded word as input, no `2 * hidden_size` concatenation trick needed like Bahdanau's `nn.RNN(2 * hidden_size, hidden_size, ...)`. Attention gets folded in *after* the RNN runs, by blending the RNN's own output with the attention context, and it's this blended `attentional_hidden` vector, not the raw RNN output, that goes into predicting the next word.

If we'd instead bolted attention onto the front of the RNN the Bahdanau way, we'd end up feeding the RNN a context vector computed from the *previous* hidden state (stale by one step) *and* separately still have the option to fuse context with hidden state again post-RNN, which is redundant, the whole point of Luong's formulation is that attention happens once, after the RNN has already updated on this step's word.


## **Implementation**

We reuse the exact same data loading, vocabulary building, and `EncoderRNN` from the main notebook, only the attention and decoder are new here. Requirements first.


In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device = {device}")


**Note:** No `torch.set_default_device()` call here on purpose, same reasoning as the main notebook, everything below places tensors explicitly with `.to(device)` / `device=device`, and setting a global default device breaks `DataLoader`'s `RandomSampler` on a CUDA machine (`RuntimeError: Expected a 'cuda' device type for generator but found 'cpu'`).

### **Data Loading**

Same as before, `data/eng-fra.txt` from [https://download.pytorch.org/tutorial/data.zip](https://download.pytorch.org/tutorial/data.zip), filtered down to short "I am / he is" style sentences under `MAX_LENGTH` words.


In [ ]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


In [ ]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

def readLangs(path: str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    pairs = [list(reversed(p)) for p in pairs]

    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs


In [ ]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


In [ ]:
PATH = r'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))


### **Encoder**

Unchanged from the main notebook, we only rewrite the decoder side here.


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden


### **Multiplicative Attention**

This is the new piece. `Wa` is the one learned weight matrix from the formula above, `score = query^T · Wa · keys`. We project the encoder outputs through `Wa` first, then batch-matrix-multiply against the query, this is mathematically the same as projecting the query and dotting with the raw keys, just cheaper to compute this way since `Wa(keys)` can be reused across the batch.

We also keep the same `Wc` fusing step as the Luong dot attention we were given, concatenate `context` with the `query`, squash through `tanh`, this gives us `attentional_hidden` (written `s~_t` in Luong's paper), which is what actually feeds the output prediction, not the raw context vector on its own.


In [ ]:
class MultiplicativeAttention(nn.Module):
    def __init__(self, hidden_size):
        super(MultiplicativeAttention, self).__init__()
        # For:
        # e_{t,i} = s_t^T Wa h_i
        self.Wa = nn.Linear(hidden_size, hidden_size, bias=False)
        # For:
        # s~_t = tanh(Wc[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)
        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)
        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)
            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """
        # Alignment scores:
        # e_{t,i} = s_t^T Wa h_i
        scores = torch.bmm(
            query,
            self.Wa(keys).transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(Wc[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights


### **Decoder**

Here's the part the teacher had us rewrite. Following the "attention happens after the RNN" ordering from the theory section above, not the Bahdanau "attention before the RNN" ordering, since bolting this attention module onto the front the Bahdanau way would double up on fusing the hidden state (once inside `Wc`, and again by concatenating into the RNN input).

`self.rnn` only ever takes the plain embedded word, so it stays `hidden_size` sized input, no `2 * hidden_size` needed here since context never gets concatenated into the RNN's input. Instead, the RNN runs first with just the embedding, and *its* output becomes the query for attention, then `self.out` predicts straight off the fused `attentional_hidden`.


In [ ]:
class MultiplicativeAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(MultiplicativeAttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.attention = MultiplicativeAttention(hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))

        # RNN runs first on just the embedded word, no context concatenated in
        rnn_output, hidden = self.rnn(embedded, hidden)

        # This step's RNN output becomes the query for attention
        query = rnn_output
        attentional_hidden, attn_weights = self.attention(query, encoder_outputs)

        # Prediction comes from the fused attentional hidden state
        output = self.out(attentional_hidden)

        return output, hidden, attn_weights


## **Training**

Same training setup as the main notebook, just pointed at `MultiplicativeAttnDecoderRNN` instead of the plain `DecoderRNN`.


In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader


In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))


In [ ]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)


In [ ]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


### **Evaluation Code**

In [ ]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')


### **Training and Evaluating**

In [ ]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = MultiplicativeAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)


In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)


## **Discussion:**
Multiplicative attention lands in the middle of the two extremes we already had, more flexible than a raw dot product since `Wa` learns how to align the query and key spaces instead of assuming they already match, but cheaper than Bahdanau's full feedforward scoring network since its just one matrix multiply.

The bigger structural change here wasn't really the scoring function though, it was moving attention from *before* the RNN step to *after* it. Bahdanau's decoder needed the context vector before the RNN could run, since it gets concatenated into the RNN's input alongside the embedded word. Luong-style decoders (dot, general, or otherwise) let the RNN run first on just the embedding, and only pull in the encoder outputs afterward to refine that RNN's output into the final prediction. Same overall goal, look back at the full input instead of relying on one static vector, just wired in a different spot in the computation.

One thing worth sanity-checking on the attention weights (`attentions` returned from `forward`), for short, formulaic sentences like the "I am / he is" style ones we filtered down to, the attention weights tend to line up fairly cleanly, since French and English word order matches closely for these simple sentence patterns. This wouldn't necessarily hold for sentences with more reordering between languages.


## **Conclusion**
We implemented Luong general (multiplicative) attention, `score = query^T · Wa · keys`, and rewrote the decoder to match Luong's ordering, RNN runs on the plain embedded word first, then attention fuses in the encoder outputs afterward through `Wc`, rather than concatenating context into the RNN's input the way the Bahdanau decoder does.

Trained with the same hidden size, batch size, and epoch count as the other two decoders on the same filtered dataset, so the outputs from `evaluateRandomly` are directly comparable across additive, dot, and multiplicative attention. Since multiplicative attention learns an explicit alignment between query and key spaces via `Wa`, unlike dot attention which has no learned parameters at all, it should generally have a bit more capacity to fit the alignment than plain dot, while still being cheaper to train than Bahdanau's full feedforward scorer.
